# **Import des modules nécessaires**

In [2]:
import pandas as pd #pour la manipulation de données
from pathlib import Path #pour la gestion des chemins de fichiers
import spacy #pour le prétraitement de texte

from bertopic import BERTopic #pour la modélisation de sujets

from umap import UMAP #pour la réduction de dimensionnalité
from hdbscan import HDBSCAN #pour le clustering de BERTopic

from sklearn.feature_extraction.text import CountVectorizer #pour la vectorisation de texte
from bertopic.vectorizers import ClassTfidfTransformer #pour la vectorisation de texte spécifique à BERTopic

from sentence_transformers import SentenceTransformer #pour les embeddings de phrases

# **Chargement du corpus de Zola et de spacy**


In [3]:
df=pd.read_csv(Path("..") /"data" /"2_processed"/ "02_corpus_zola.csv", encoding="utf-8",)
df.head()


,roman,annee,ordre_romans,paquet_id,texte,nb_mots
0,1865 La confession de Claude.,1865,1,1,"Voici l’hiver: l’air, au matin, devient plus f...",401
1,1865 La confession de Claude.,1865,1,2,Le grillon chantait; le souffle harmonieux des...,336
2,1865 La confession de Claude.,1865,1,3,"Pars cependant, puisque tu as soif de la vie. ...",248
3,1865 La confession de Claude.,1865,1,4,"Ai-je eu trop de confiance en ma force, ma pla...",336
4,1865 La confession de Claude.,1865,1,5,Le premier rayon a chassé le cauchemar de ma v...,359


In [4]:
df.shape

(10638, 6)

# **Traitement du Corpus de Zola**

In [6]:
# Chargement du modèle avec désactivation du 'parser' syntaxique pour gagner en vitesse
# On garde impérativement 'ner' pour repérer les personnages/lieux et 'lemmatizer'
nlp = spacy.load("fr_core_news_lg", disable=["parser"])
nlp.max_length = 2_000_000  

#on convertit en liste
textes_bruts = df["texte"].astype(str).tolist()

textes_nettoyes = []

# Utilisation de nlp.pipe pour traiter les textes par blocs (très rapide)
for doc in nlp.pipe(textes_bruts, batch_size=50): 
    tokens = [] # Liste pour stocker les tokens nettoyés
    
    for token in doc:
        lemme = token.lemma_.lower() # Obtenir le lemme du token en minuscules
        
        if (
            not token.is_stop # Ignorer les stop words spaCy par défaut
            and not token.is_punct # Ignorer la ponctuation
            and not token.like_num # Ignorer les chiffres
            and not token.is_space # Ignorer les espaces vides
            and token.ent_type_ not in ['PER', 'LOC', 'ORG'] # Ignorer les Personnages, Lieux et Organisations
            and token.pos_ in {"NOUN", "ADJ", "VERB"}  # Garder Noms, Adjectifs ET Verbes
            and len(lemme) > 2 # Ignorer les mots de 1 ou 2 lettres
        ):
            tokens.append(lemme)
            
    # Rejoindre les tokens validés et les ajouter à la liste finale
    textes_nettoyes.append(" ".join(tokens))

# Application de la liste nettoyée à la nouvelle colonne du DataFrame
df["phrases_lemm"] = textes_nettoyes

# Affichage du résultat
df[["phrases_lemm"]].head()

,phrases_lemm
0,hiver air matin devenir frais mettre manteau b...
1,grillon chanter souffle harmonieux nuit bercer...
2,soif vie projet soi ferme loyal action rêve vi...
3,confiance force place rêver côté jour naître p...
4,rayon chasser cauchemar veille obstacle accept...


## **1) Choix du modèle d'embedding**

Ici je vais choisir un modèle d'embedding pré-entraîné pour transformer les textes en vecteurs numériques. Je vais utiliser un modèle de la bibliothèque Sentence Transformers, qui est compatible avec BERTopic.

In [7]:
embedding_model = SentenceTransformer(
    "dangvantuan/sentence-camembert-base"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [8]:
print("Génération des embeddings sémantiques...")
embeddings = embedding_model.encode(df['texte'].tolist(), show_progress_bar=True)

Génération des embeddings sémantiques...


Batches:   0%|          | 0/333 [00:00<?, ?it/s]

## **2) Pipeline de Traitement**

### 1) HDBSCAN et UMAP

In [9]:
hdbscan_model = HDBSCAN(
    min_cluster_size=30,
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)


umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

### 2) creation des stop words + vectorisation TF-IDF

In [11]:
stop_word = [
    # noms familiaux centraux
    "rougon", "macquart", "saccard", "mouret", "lantier", "coupeau", "maheu", "fouan", "quenu", "chanteau", "josserand", "deberle",
    "raquin", "lorilleux", "buteau", "roubaud", "baudu", "gervaise",

    # Prénoms / personnages très fréquents
    "pierre", "jean", "jacques", "marie", "madeleine", "pauline", "hélène", "jeanne", "claude", "marius", "lazare", "guillaume",
    "laurent", "philippe", "thérèse", "albine", "maurice", "marthe", "maxime", "daniel", "camille", "serge", "octave", "florent",
    "félicité", "louise", "silvère", "caroline", "lisa", "laurence", "miette", "simon", "josine", "clotilde", "véronique", "juliette",
    "rosalie", "mathieu", "charles", "françoise", "lise", "séverine", "henriette", "georges", "angélique", "adèle", "pascal", "lucie", 
    "martine", "virginie", "suzanne", "valentine", "olympe", "lucien", "armande", "constance", "berthe", "sidonie", "benedetta",

    # prénom / personnages complémentaires
    "luc", "geneviève", "catherine", "chaval", "adélaïde", "antoine", "auguste", "marianne", "clorinde", "normande", "saget", "mlle saget",
    "delhomme", "ragu", "trouche", "sandoz", "mahoudeau", "bonneville", "rastoil", "rambaud", "fenil", "surin", "malignon", "faloise",
    "lambesc", "mignot", "baugé", "morange", "bouchard", "teuse", "pâquerette", "paquerette", "duparque", "mme duparque", "guersaint",
    "aurélie", "aurelie", "mme aurélie", "mme aurelie", "lecœur", "lecoeur", "mme lecœur", "mme lecoeur", "granoux", "bourron",
    "lerat", "mme lerat", "duveyrier", "bachelard", "gueulin", "delaherche", "beauclair", "lison", "orlando", "delbos",
    "boisgelin", "fernande", "sébastien", "sebastien", "mazaud", "dubuche", "moreux", "vigneron", "bourrette", "satan",
    "archangias", "férat", "ferat", "barbue", "trouille", "laveuve",

    # Personnages secondaires / noms très discriminants
    "fauchery", "muffat", "bordenave", "zoé", "mignon", "faujas", "mathéus", "trublot", "fagerolles", "cazalis", "poizat", "jordan",
    "vandeuvres", "condamin", "paloque", "charbonnel", "gilquin", "marjolin", "campardon", "toussaint", "grivet", "denizet", "kahn",
    "hutin", "bonnaire", "laboque", "marsy", "nanet", "favier", "deloche", "goujet", "cadine", "chouteau", "rochas", "lepailleur",
    "séguin", "gagnière", "hourdequin", "macqueron", "théodose", "savin", "gérard", "larsonneau", "morfain", "lorin", "salvat",
    "boche", "gavard", "martelly",

    # Groupes nominaux religieux / institutionnels
    "sœur hyacinthe", "soeur hyacinthe", "mère chantemesse", "mere chantemesse",

     # Formes d’adresse / famille
    "monsieur", "madame", "mademoiselle", "mme", "mlle",
    "maman", "papa", "père", "pere", "mère", "mere",
    "frère", "frere", "frères", "freres", "sœur", "soeur",
    "sœurette", "soeurette", "tante", "oncle", "cousin",
    "cousine", "époux", "epoux", "épouse", "epouse",
    "nièce", "niece",

    # Mots très fréquents et peu informatifs
    "le", "la", "les", "un", "une", "des", "du", "de", "d", "au", "aux",
    "ce", "cet", "cette", "ces", "son", "sa", "ses", "leur", "leurs",
    "je", "tu", "il", "elle", "nous", "vous", "ils", "elles",
    "me", "te", "se", "moi", "toi", "lui", "eux",
    "que", "qui", "quoi", "dont", "où", "et", "ou", "mais", "donc",
    "or", "ni", "car", "ne", "pas", "plus", "moins", "très",
    "dans", "sur", "sous", "avec", "sans", "pour", "par", "comme",
    "chez", "vers", "entre", "contre", "depuis", "pendant",
    "tout", "tous", "toute", "toutes", "même", "autre", "autres",
    "être", "avoir", "faire", "dire", "aller", "voir", "savoir",
    "pouvoir", "vouloir", "falloir", "devoir", "venir", "prendre",
    "mettre", "donner", "trouver", "passer", "rester", "sembler",
    "encore", "toujours", "jamais", "bien", "mal", "peu", "assez",
    "alors", "puis", "enfin", "aussi", "ainsi", "déjà", "là",
    "ici", "cela", "ça", "ceci", "celui", "celle", "ceux", "celles"
]

### 3) CountVectorizer et ClassTfidfTransformer avec des stop words personnalisés 

In [14]:
df.head()

,roman,annee,ordre_romans,paquet_id,texte,nb_mots,phrases_lemm
0,1865 La confession de Claude.,1865,1,1,"Voici l’hiver: l’air, au matin, devient plus f...",401,hiver air matin devenir frais mettre manteau b...
1,1865 La confession de Claude.,1865,1,2,Le grillon chantait; le souffle harmonieux des...,336,grillon chanter souffle harmonieux nuit bercer...
2,1865 La confession de Claude.,1865,1,3,"Pars cependant, puisque tu as soif de la vie. ...",248,soif vie projet soi ferme loyal action rêve vi...
3,1865 La confession de Claude.,1865,1,4,"Ai-je eu trop de confiance en ma force, ma pla...",336,confiance force place rêver côté jour naître p...
4,1865 La confession de Claude.,1865,1,5,Le premier rayon a chassé le cauchemar de ma v...,359,rayon chasser cauchemar veille obstacle accept...


In [15]:
vectorizer_model = CountVectorizer(
    stop_words=stop_word,
    min_df=5, #
    max_df=0.50 #   
)

ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True
)


topic_model = BERTopic(
    language="french",
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=False,
    verbose=True
)
topics, probs = topic_model.fit_transform(df["phrases_lemm"].tolist(), embeddings= embeddings)

2026-06-03 12:24:20,538 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-03 12:24:36,578 - BERTopic - Dimensionality - Completed ✓
2026-06-03 12:24:36,579 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-03 12:24:37,027 - BERTopic - Cluster - Completed ✓
2026-06-03 12:24:37,032 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-03 12:24:37,871 - BERTopic - Representation - Completed ✓


## **3) Topics Présent**

In [16]:
topic_info = topic_model.get_topic_info()
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,7354,-1_juif_balle_loge_embêter,"[juif, balle, loge, embêter, évêque, moral, po...",[soir avoir coup cœur boulevard voir venir lar...
1,0,1143,0_insurgé_cortège_rive_batterie,"[insurgé, cortège, rive, batterie, kilomètre, ...",[étage maison flanc colline ville énorme bloc ...
2,1,385,1_noiraud_voluptueux_union_anéantissement,"[noiraud, voluptueux, union, anéantissement, l...",[long après-midi accablement venir passer heur...
3,2,250,2_catholicisme_dogme_moral_démocratie,"[catholicisme, dogme, moral, démocratie, socia...",[travailleur mécontent plaindre misère aggrave...
4,3,200,3_étreindre_haïr_bohémien_apaisement,"[étreindre, haïr, bohémien, apaisement, certif...",[rêv exil exil permettre vivre sein désirer em...
5,4,197,4_prélat_évêque_audience_vicaire,"[prélat, évêque, audience, vicaire, eminence, ...",[reprendre livre faire annoncer air riant main...
6,5,171,5_jury_jésuite_juif_audience,"[jury, jésuite, juif, audience, juré, municipa...",[attente exaspérer attente prolonger suppositi...
7,6,158,6_veilleuse_pantoufle_accouchée_deberl,"[veilleuse, pantoufle, accouchée, deberl, poti...",[jusqu’ heure matin frissonnante rester camiso...
8,7,138,7_oie_boudin_rigoler_forgeron,"[oie, boudin, rigoler, forgeron, fourchette, c...",[coupeau déjeuner heure charcuterie fourneau o...
9,8,123,8_mercerie_hall_confection_gant,"[mercerie, hall, confection, gant, dévisager, ...",[passer lever suivre conseil ancien condiscipl...


In [18]:
# Récupération de la dimension temporelle
timestamps = df['annee'].tolist()

# Génération des topics dans le temps
topics_over_time = topic_model.topics_over_time(
    df['phrases_lemm'].tolist(), 
    timestamps, 
    nr_bins=20 # Optionnel : à ajuster selon le nombre d'années distinctes de ton corpus
)

# Visualisation interactive de l'évolution des 10 premiers topics
topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=10)

19it [00:01, 15.06it/s]


In [19]:
for topic_id in topic_info["Topic"].head(15):
    if topic_id != -1:
        print("\nTOPIC", topic_id)
        print(topic_model.get_topic(topic_id)[:15])


TOPIC 0
[('insurgé', np.float64(0.01666140829867937)), ('cortège', np.float64(0.0124784118893224)), ('rive', np.float64(0.012081052632016198)), ('batterie', np.float64(0.01186910434022644)), ('kilomètre', np.float64(0.011573083300353653)), ('balle', np.float64(0.011207425058890204)), ('division', np.float64(0.011183144227312994)), ('vallée', np.float64(0.011004849969838752)), ('temple', np.float64(0.010901192690013183)), ('panique', np.float64(0.010543802820705776))]

TOPIC 1
[('noiraud', np.float64(0.01863572631117798)), ('voluptueux', np.float64(0.01489993489943332)), ('union', np.float64(0.014568936680831148)), ('anéantissement', np.float64(0.013753786061015374)), ('laboratoire', np.float64(0.013682861491959915)), ('revivre', np.float64(0.013675883611122664)), ('châtiment', np.float64(0.013032231060264047)), ('insomnie', np.float64(0.012920030193352802)), ('calèche', np.float64(0.0124712409417365)), ('fantôme', np.float64(0.011990010596294431))]

TOPIC 2
[('catholicisme', np.float6